# ĐỀ XUẤT (PROPOSAL)

## Dự đoán giá cổ phiếu VN-Index dựa trên biến động giá trong quá khứ

| | |
|---|---|
| **Môn học** | Học Máy (Machine Learning) |
| **Trường** | Đại học Sư phạm Kỹ thuật TP.HCM (HCMUTE) |
| **Sinh viên thực hiện** | Bá Hoài Sơn — Bùi Thanh Tú |
| **Mã thử nghiệm** | `VCB.VN` (Vietcombank — Yahoo Finance) |
| **Loại bài toán** | Hồi quy có giám sát trên chuỗi thời gian |

---

## 4.1. Giới thiệu về bài toán và dữ liệu

### 4.1.1. Bối cảnh

Thị trường chứng khoán Việt Nam được đo lường bởi chỉ số **VN-Index** — phản
ánh xu hướng chung của các cổ phiếu niêm yết trên Sở Giao dịch Chứng khoán
TP.HCM. Việc dự đoán biến động giá cổ phiếu trong tương lai có ý nghĩa lớn:

- **Nhà đầu tư cá nhân và tổ chức** — hỗ trợ ra quyết định mua/bán.
- **Quản trị danh mục** — tối ưu phân bổ tài sản, hedge rủi ro.
- **Nghiên cứu kinh tế** — kiểm chứng Hiệu Quả Thị Trường (EMH).

### 4.1.2. Câu hỏi nghiên cứu

> **Q1.** Có thể dự đoán biến động giá VCB ở khung trung hạn (1 tháng giao
> dịch, ≈ 20 phiên) từ các đặc trưng kỹ thuật rút từ giá lịch sử không?
>
> **Q2.** Tín hiệu xu hướng có thay đổi theo *horizon* dự đoán hay không?
> Khung 1 ngày, 5 ngày, 20 ngày, 60 ngày có cùng độ khó?
>
> **Q3.** Linear Regression có đủ — hay phải dùng KNN / Random Forest /
> Voting Ensemble mới đạt DirAcc > 50 %?

### 4.1.3. Tập dữ liệu

Dữ liệu giá lịch sử của **VCB** được tải qua thư viện
[`yfinance`](https://pypi.org/project/yfinance/) (Yahoo Finance API):
- 4.209 phiên ngày (2009-06-30 → 2026-05-15) — dùng huấn luyện chính.
- 3.464 phiên giờ — phục vụ phân tích bổ sung.

Mỗi quan sát gồm `Date`, `Open`, `High`, `Low`, `Close`, `Volume`,
`Interval`, `Ticker`/`Symbol`. Tổ hợp đầy đủ biến phân loại + rời rạc +
liên tục, thỏa yêu cầu môn học.

In [1]:
import sys, pandas as pd
sys.path.insert(0, '../scripts')

df_raw = pd.read_csv('../data/vcb_stock.csv', parse_dates=['Date'])
print('Tổng số dòng:', len(df_raw))
print(df_raw.groupby('Interval').size().to_string())
print('Khoảng:', df_raw['Date'].min(), '→', df_raw['Date'].max())
df_raw.head()

Tổng số dòng: 7673
Interval
1d    4209
1h    3464
Khoảng: 2009-06-30 00:00:00 → 2026-05-15 07:00:00+00:00


,Date,Close,High,Low,Open,Volume,Interval,Ticker,Symbol
0,2023-07-31 02:00:00+00:00,61404.683594,62341.136719,61270.902344,62207.359375,0,1h,VCB,VCB.VN
1,2023-07-31 03:00:00+00:00,61471.570312,61605.351562,61337.792969,61404.683594,128504,1h,VCB,VCB.VN
2,2023-07-31 04:00:00+00:00,61404.683594,61538.460938,61404.683594,61471.570312,54401,1h,VCB,VCB.VN
3,2023-07-31 06:00:00+00:00,61538.460938,61672.242188,61404.683594,61404.683594,327312,1h,VCB,VCB.VN
4,2023-07-31 07:00:00+00:00,61270.902344,62073.578125,61270.902344,61538.460938,0,1h,VCB,VCB.VN


## 4.2. Kế hoạch phân tích dữ liệu

### 4.2.1. Định nghĩa input / output

Sau giai đoạn thí nghiệm sơ bộ (xem Milestone), nhóm nhận ra dự đoán giá
*hôm sau* gặp **cạm bẫy ngoại suy** và DirAcc 1-day ~ 50 % (mức ngẫu nhiên).
Vì vậy nhóm chọn target chính là **log-return 20 phiên tới** — khung 1
tháng giao dịch, phổ biến trong chiến lược *swing trading* và là khung mà
tín hiệu xu hướng vượt mức ngẫu nhiên có ý nghĩa.

$$\boxed{\;\hat r_{t+h} = f(\mathbf x_t),\quad \hat C_{t+h} = C_t\,e^{\hat r_{t+h}},\quad h = 20\;}$$

| Thành phần | Biến |
|------------|------|
| **Output (Y)** | `Target_Return` = $\log(C_{t+20}/C_t)$ |
| **Input (X) — 25 đặc trưng** | xem bảng dưới |

### 4.2.2. Bộ 25 đặc trưng stationary

| Nhóm | Đặc trưng | Vai trò |
|------|-----------|---------|
| Lợi suất quá khứ | `Return_{1,2,3,5,10,20,60}` | Động lượng đa khung |
| Tỷ lệ với MA | `MA{5,10,20,50,100}_Ratio` | Xu hướng |
| Volatility | `Vol_{5,10,20}` | Rủi ro / dao động |
| Chỉ báo kỹ thuật | `RSI_14`, `MACD`, `MACD_Signal`, `MACD_Hist`, `Bollinger_b` | Quá mua / quá bán, đảo chiều |
| Biên độ phiên | `HL_Range`, `OC_Range` | Lực giằng co |
| Khối lượng | `Vol_Change`, `Volume_MA20_Ratio` | Dòng tiền |
| Xu hướng dài hạn | `Trend_MA50_200` | Golden cross / death cross |

### 4.2.3. Độ đo

| Metric | Diễn giải |
|--------|-----------|
| **MAE / RMSE** | Sai số trên log-return |
| **R²** | Tỷ lệ phương sai được giải thích |
| **DirAcc(%)** | Tỷ lệ đoán đúng chiều (lên/xuống) — **chỉ số chính** |
| **DirAcc_filt(%)** | DirAcc loại 10 % phiên biến động nhỏ nhất (lọc nhiễu) |
| **MAPE(%)** trên giá VND | Sai số tương đối khi back-transform về giá |

**Chia train/test:** chronological 80/20 (không xáo trộn) — phản ánh kịch
bản triển khai thực tế.

### 4.2.4. Bốn mô hình thử nghiệm

1. **Linear Regression** — baseline tuyến tính.
2. **K-Nearest Neighbors (k = 25)** — phi tham số, học theo các phiên giống.
3. **Random Forest (500 cây, max_depth = 6, min_samples_leaf = 20)** —
   phi tuyến, robust.
4. **Voting Ensemble** — trung bình dự đoán của 3 mô hình trên.

### 4.2.5. Kế hoạch thực hiện và phân công

| Tuần | Mục tiêu | Phụ trách chính |
|------|----------|-----------------|
| 1 | Tải dữ liệu qua `yfinance`, EDA cơ bản | Bá Hoài Sơn |
| 2 | Feature engineering, train LR + KNN | Bá Hoài Sơn |
| 3 | Train RF + Ensemble, sweep đa horizon | Bùi Thanh Tú |
| 4 | So sánh, biểu đồ, Milestone | Cả nhóm |
| 5 | Report + Presentation | Cả nhóm |

**Phân công cụ thể**

| Thành viên | Đảm nhiệm |
|------------|-----------|
| **Bá Hoài Sơn** | Thu thập dữ liệu; EDA; Linear Regression; KNN; biểu đồ giá & phân phối; Proposal; Presentation. |
| **Bùi Thanh Tú** | Feature engineering nâng cao (MACD, Bollinger, longer momentum); Random Forest; Voting Ensemble; sweep đa horizon; Milestone; Report. |

## 4.3. Kết quả kỳ vọng

- **DirAcc trên horizon 20 phiên ≥ 52 %** cho ít nhất 1 mô hình (vượt mức
  ngẫu nhiên có ý nghĩa).
- **MAPE giá** dưới 5 % (1 tháng).
- **Phát hiện đa horizon**: DirAcc tăng theo horizon — xác nhận tín hiệu xu
  hướng trung hạn dự đoán được, còn ngắn hạn bị nhiễu áp đảo.

---
*Kết thúc Đề xuất.*